In [1]:
# https://github.com/prophesee-ai/openeb/blob/5.2.0/sdk/modules/stream/python/samples/metavision_sdk_get_started/metavision_sdk_get_started_v2.py
import evcam  # before any metavision import
import numpy as np
from PIL import Image
from metavision_sdk_core import BaseFrameGenerationAlgorithm
from metavision_sdk_stream import Camera, CameraStreamSlicer, SliceCondition


In [2]:
with evcam.quiet_stderr(): 
    camera = Camera.from_first_available()


slicer = CameraStreamSlicer(camera.move(), slice_condition=SliceCondition.make_n_us(10_000))
width = slicer.camera().width()
height = slicer.camera().height()
print(f"{width} x {height}")


RuntimeError: 
------------------------------------------------
Metavision SDK Stream exception

Error 101001: Camera not found. Check that a camera is plugged on your system and retry.
------------------------------------------------


In [ ]:
global_counter = 0  # This will track how many events we processed
global_max_t = 0  # This will track the highest timestamp we processed

for i, slice in enumerate(slicer):
    print("----- New event slice! -----")
    if slice.events.size == 0:
        print("The current event slice is empty.")
    else:
        min_t = slice.events['t'][0]   # Get the timestamp of the first event of this slice
        max_t = slice.events['t'][-1]  # Get the timestamp of the last event of this callback
        global_max_t = max_t  # Events are ordered by timestamp, so the current last event has the highest timestamp

        counter = slice.events.size  # Local counter
        global_counter += counter  # Increase global counter

        print(f"There were {counter} events in this event slice.")
        print(f"There were {global_counter} total events up to now.")
        print(f"The current event slice included events from {min_t} to {max_t} microseconds.")
        print("----- End of the event slice! -----")
    if i == 9:  # the live camera never ends, so stop after 10 slices (100 ms)
        break


In [ ]:
duration_seconds = global_max_t / 1.0e6
print(f"There were {global_counter} events in total.")
print(f"The total duration was {duration_seconds:.2f} seconds.")
if duration_seconds >= 1:  # No need to print this statistics if the total duration was too short
    print(f"There were {global_counter / duration_seconds :.2f} events per second on average.")


In [ ]:
frame = np.zeros((height, width, 3), np.uint8)
slice = next(iter(slicer))
BaseFrameGenerationAlgorithm.generate_frame(slice.events, frame)
Image.fromarray(frame[..., ::-1])  # BGR -> RGB


In [ ]:
del slicer, slice  # releases the camera so other scripts (or re-running cell 2) can open it
